# Sprint 1 — Component 4 Lung Mask Decoder Training

Trains **only** the MedSAM ViT-B mask decoder on Montgomery + Shenzhen lung annotations.
The image encoder and prompt encoder stay completely frozen — only the decoder weights change.

## Datasets to attach before running

| Kaggle dataset slug | Confirmed mount path | Purpose |
|---|---|---|
| `iahmedhabib/montgomery` | `/kaggle/input/datasets/iahmedhabib/montgomery/montgomery` | CXR images + left/right lung masks |
| `iahmedhabib/shehzhenn` | `/kaggle/input/datasets/iahmedhabib/shehzhenn/shenzhen` | CXR images (`images/images/`) + masks (`mask/mask/`) |
| `iahmedhabib/medsam-vit-b` | `/kaggle/input/datasets/iahmedhabib/medsam-vit-b/medsam_vit_b.pth` | MedSAM ViT-B pretrained checkpoint |

## What this trains

- **Model**: MedSAM ViT-B mask decoder (~4 M trainable params; encoder frozen)
- **Loss**: BCE + Dice on 256×256 low-res masks
- **Schedule**: 3-epoch linear warmup → cosine decay to floor LR over 60 epochs
- **Augmentation**: horizontal flip, ±10° rotation (image+mask jointly), brightness jitter, Gaussian noise
- **Dataset mixing**: interleaved by dataset before DataLoader shuffle — every batch is mixed
- **Training data**: 572 samples (Montgomery 109 + Shenzhen 463)
- **Val data**: 132 samples (Montgomery 29 + Shenzhen 103)

## Achieved results (confirmed on Kaggle T4)

| Metric | Before Sprint 1 | After Sprint 1 |
|---|---|---|
| Montgomery val Dice | 0.42 | **0.8901** |
| Shenzhen val Dice | 0.51 | **0.9594** |
| Overall val Dice | — | **0.9442** (best at epoch 38/60) |

Estimated wall-clock: **~90 minutes on a T4 GPU** (early stop fired at epoch 42).

In [ ]:
# ── Cell 1: Clone repo + install dependencies ────────────────────────────────
import subprocess, sys, os
from pathlib import Path

REPO_URL = 'https://github.com/mabdullahi7780/dl-project-codebase.git'
REPO_DIR = Path('/kaggle/working/repo')

if not REPO_DIR.exists():
    print('Cloning repo...')
    subprocess.run(['git', 'clone', '--depth', '1', REPO_URL, str(REPO_DIR)], check=True)
else:
    print('Repo already exists — pulling latest...')
    subprocess.run(['git', '-C', str(REPO_DIR), 'pull', '--ff-only'], check=True)

os.chdir(str(REPO_DIR))
sys.path.insert(0, str(REPO_DIR))

print('Installing dependencies...')
subprocess.run(
    [sys.executable, '-m', 'pip', 'install', '-q', '-r', 'requirements.txt'],
    check=True
)

import torch
print(f'PyTorch {torch.__version__}')
print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)} ({torch.cuda.get_device_properties(0).total_memory // 1024**3} GB)')
subprocess.run(['git', 'branch', '--show-current'], cwd=str(REPO_DIR))

In [ ]:
# ── Cell 2: Discover dataset paths ───────────────────────────────────────────
#
# Confirmed working mount paths (Kaggle datasets tab, May 2026):
#   Montgomery : /kaggle/input/datasets/iahmedhabib/montgomery/montgomery/
#   Shenzhen   : /kaggle/input/datasets/iahmedhabib/shehzhenn/shenzhen/
#                  images live at  shenzhen/images/images/*.png
#                  masks live at   shenzhen/mask/mask/*.png
#   MedSAM     : /kaggle/input/datasets/iahmedhabib/medsam-vit-b/medsam_vit_b.pth

from pathlib import Path

def _find(*candidates):
    """Return the first existing path, or None."""
    for p in candidates:
        p = Path(p)
        if p.exists():
            return p
    return None

# ── MedSAM pretrained backbone — required ────────────────────────────────────
MEDSAM_CKPT = _find(
    '/kaggle/input/datasets/iahmedhabib/medsam-vit-b/medsam_vit_b.pth',  # confirmed
    '/kaggle/input/medsam-vit-b/medsam_vit_b.pth',
    '/kaggle/input/medsam-vit-b/MedSAM/medsam_vit_b.pth',
    REPO_DIR / 'checkpoints/medsam/medsam_vit_b.pth',
)

# ── Montgomery root (contains CXR_png/ + ManualMask/) ────────────────────────
MONTGOMERY_ROOT = _find(
    '/kaggle/input/datasets/iahmedhabib/montgomery/montgomery',           # confirmed
    '/kaggle/input/montgomery/MontgomerySet',
    '/kaggle/input/montgomery/montgomery/MontgomerySet',
    '/kaggle/input/montgomery',
)

# ── Shenzhen root (images at root/images/images/, masks at root/mask/mask/) ──
SHENZHEN_ROOT = _find(
    '/kaggle/input/datasets/iahmedhabib/shehzhenn/shenzhen',             # confirmed
    '/kaggle/input/shehzhenn/ChinaSet_AllFiles',
    '/kaggle/input/shehzhenn/shenzhen/ChinaSet_AllFiles',
    '/kaggle/input/shehzhenn',
)

print('Path discovery:')
print(f'  MEDSAM_CKPT    : {MEDSAM_CKPT    or "NOT FOUND"}')
print(f'  MONTGOMERY_ROOT: {MONTGOMERY_ROOT or "NOT FOUND"}')
print(f'  SHENZHEN_ROOT  : {SHENZHEN_ROOT  or "NOT FOUND"}')

assert MEDSAM_CKPT is not None, (
    'MedSAM checkpoint not found. '
    'Attach iahmedhabib/medsam-vit-b and confirm the file is medsam_vit_b.pth'
)
assert MONTGOMERY_ROOT is not None or SHENZHEN_ROOT is not None, (
    'No annotated dataset found. Attach iahmedhabib/montgomery or iahmedhabib/shehzhenn.'
)

In [ ]:
# ── Cell 3: Build component4_manifest.csv ────────────────────────────────────
#
# Columns: sample_id, image_path, mask_path, dataset, split
#
# Montgomery: left+right lung mask PNGs merged on-the-fly via pipe separator.
# Shenzhen  : single combined mask per image.
#             Confirmed structure: images/images/*.png and mask/mask/*.png
#             Mask filenames: {stem}_mask.png  (fallback: {stem}.png)
#
# Split: deterministic 80/20 by hash(stem) % 5 — same 20% always goes to val.

import csv
from pathlib import Path
from collections import Counter

OUTPUT_DIR = Path('/kaggle/working/component4_out')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
MANIFEST_PATH = OUTPUT_DIR / 'component4_manifest.csv'

rows = []

def _split(stem: str) -> str:
    return 'val' if abs(hash(stem)) % 5 == 0 else 'train'

# ── Montgomery ───────────────────────────────────────────────────────────────
if MONTGOMERY_ROOT is not None:
    mont_root = Path(MONTGOMERY_ROOT)

    img_dir = _find(
        mont_root / 'CXR_png',
        mont_root.parent / 'CXR_png',
    )
    left_dir = _find(
        mont_root / 'ManualMask' / 'leftMask',
        mont_root.parent / 'ManualMask' / 'leftMask',
    )
    right_dir = _find(
        mont_root / 'ManualMask' / 'rightMask',
        mont_root.parent / 'ManualMask' / 'rightMask',
    )

    if img_dir and left_dir and right_dir:
        for img in sorted(img_dir.glob('*.png')):
            stem = img.stem
            left  = left_dir  / f'{stem}.png'
            right = right_dir / f'{stem}.png'
            if left.exists() and right.exists():
                rows.append({
                    'sample_id' : stem,
                    'image_path': str(img),
                    'mask_path' : f'{left}|{right}',
                    'dataset'   : 'montgomery',
                    'split'     : _split(stem),
                })
        n = sum(1 for r in rows if r['dataset'] == 'montgomery')
        print(f'Montgomery: {n} images')
        print(f'  img_dir   : {img_dir}')
        print(f'  left_dir  : {left_dir}')
        print(f'  right_dir : {right_dir}')
    else:
        missing = [name for name, d in [('img_dir', img_dir), ('left_dir', left_dir), ('right_dir', right_dir)] if d is None]
        print(f'Montgomery: skipped — could not find: {missing}')
        print(f'  mont_root contents: {[p.name for p in mont_root.iterdir()]}')

# ── Shenzhen ─────────────────────────────────────────────────────────────────
# Confirmed layout on iahmedhabib/shehzhenn (May 2026):
#   shenzhen/images/images/*.png   — CXR images
#   shenzhen/mask/mask/*.png       — lung masks ({stem}_mask.png)
#   shenzhen/shenzhen_metadata.csv — metadata (not used here)
if SHENZHEN_ROOT is not None:
    shen_root = Path(SHENZHEN_ROOT)

    img_dir = _find(
        shen_root / 'images' / 'images',   # confirmed double-nested
        shen_root / 'images',
        shen_root / 'CXR_png',
        shen_root.parent / 'CXR_png',
    )
    mask_dir = _find(
        shen_root / 'mask' / 'mask',       # confirmed double-nested
        shen_root / 'mask',
        shen_root / 'Masks',
        shen_root.parent / 'Masks',
    )

    if img_dir and mask_dir:
        n_before = len(rows)
        for img in sorted(img_dir.glob('*.png')):
            stem = img.stem
            mask = mask_dir / f'{stem}_mask.png'
            if not mask.exists():
                mask = mask_dir / f'{stem}.png'
            if mask.exists():
                rows.append({
                    'sample_id' : stem,
                    'image_path': str(img),
                    'mask_path' : str(mask),
                    'dataset'   : 'shenzhen',
                    'split'     : _split(stem),
                })
        n = len(rows) - n_before
        print(f'Shenzhen: {n} images')
        print(f'  img_dir  : {img_dir}')
        print(f'  mask_dir : {mask_dir}')
    else:
        missing = [name for name, d in [('img_dir', img_dir), ('mask_dir', mask_dir)] if d is None]
        print(f'Shenzhen: skipped — could not find: {missing}')
        print(f'  shen_root contents: {[p.name for p in shen_root.iterdir()]}')

# ── Write manifest ────────────────────────────────────────────────────────────
assert rows, 'No image+mask pairs found — check the path discovery output above.'

with open(MANIFEST_PATH, 'w', newline='') as f:
    writer = csv.DictWriter(f, fieldnames=['sample_id', 'image_path', 'mask_path', 'dataset', 'split'])
    writer.writeheader()
    writer.writerows(rows)

train_rows = [r for r in rows if r['split'] == 'train']
val_rows   = [r for r in rows if r['split'] == 'val']

print(f'\nManifest: {MANIFEST_PATH}')
print(f'  Train : {len(train_rows)} — {dict(Counter(r["dataset"] for r in train_rows))}')
print(f'  Val   : {len(val_rows)}   — {dict(Counter(r["dataset"] for r in val_rows))}')

In [ ]:
# ── Cell 4: Train Component 4 ─────────────────────────────────────────────────
#
# Only the MedSAM mask decoder is trained. The frozen ViT-B encoder produces
# a [B, 256, 64, 64] embedding from each 1024×1024 CXR. The decoder maps that
# embedding to a [B, 1, 256, 256] lung mask logit. Loss = BCE + Dice.
#
# Sprint 1 changes vs previous run:
#   1. Cosine LR: warmup 0→3e-4 over 3 epochs, then cosine to 3e-6 by epoch 60.
#   2. Epochs 40→60, patience 8→15.
#   3. ±10° rotation applied identically to image + mask.
#   4. Gaussian noise on image only (std=0.02).
#   5. Round-robin dataset interleave so first epoch batches are always mixed.
#   6. Threshold sweep after training — prints optimal binarisation cutoff.

import subprocess, sys, os
from pathlib import Path

REPO_DIR   = Path('/kaggle/working/repo')
OUTPUT_DIR = Path('/kaggle/working/component4_out')

env = os.environ.copy()
env['COMPONENT4_TRAIN_MANIFEST'] = str(OUTPUT_DIR / 'component4_manifest.csv')
env['COMPONENT4_VAL_MANIFEST']   = str(OUTPUT_DIR / 'component4_manifest.csv')
env['COMPONENT4_SAVE_DIR']       = str(OUTPUT_DIR)

# Symlink the MedSAM checkpoint to the relative path the config expects.
ckpt_link = REPO_DIR / 'checkpoints' / 'medsam' / 'medsam_vit_b.pth'
ckpt_link.parent.mkdir(parents=True, exist_ok=True)
if not ckpt_link.exists():
    ckpt_link.symlink_to(MEDSAM_CKPT)
    print(f'Symlinked: {ckpt_link} -> {MEDSAM_CKPT}')
else:
    print(f'Checkpoint already linked: {ckpt_link}')

cmd = [
    sys.executable, '-m', 'src.training.train_component4_lung',
    '--config', str(REPO_DIR / 'configs' / 'component4_lung.yaml'),
]
print(f'\nRunning: {" ".join(cmd)}')
print('─' * 60)

result = subprocess.run(cmd, env=env, cwd=str(REPO_DIR))

print('─' * 60)
if result.returncode != 0:
    raise RuntimeError('Training failed — read the output above for the error.')
print('Training finished successfully.')

In [ ]:
# ── Cell 5: Training summary ──────────────────────────────────────────────────
import json, torch
from pathlib import Path

OUTPUT_DIR = Path('/kaggle/working/component4_out')
best_ckpt  = OUTPUT_DIR / 'component4_mask_decoder.pt'
history    = OUTPUT_DIR / 'training_history.jsonl'

for p in [best_ckpt, OUTPUT_DIR / 'last_component4_mask_decoder.pt', history]:
    tag = f'{p.stat().st_size // 1024:>8} KB' if p.exists() else '    MISSING'
    print(f'  {p.name:<45} {tag}')

if history.exists():
    records = [json.loads(l) for l in history.read_text().strip().splitlines()]
    print(f'\nEpochs run: {len(records)}')
    print(f'{"Epoch":>6} {"LR":>10} {"Train Dice":>12} {"Val Dice":>10} {"Val IoU":>9}')
    print('─' * 54)
    show = (set(range(min(3, len(records))))
            | set(range(0, len(records), 10))
            | set(range(max(0, len(records)-3), len(records))))
    for i, r in enumerate(records):
        if i in show:
            lr = r.get('lr', float('nan'))
            print(f'{r["epoch"]:>6} {lr:>10.2e} {r["train"]["dice"]:>12.4f} '
                  f'{r["val"]["dice"]:>10.4f} {r["val"]["iou"]:>9.4f}')
    best = max(records, key=lambda r: r['val']['dice'])
    print(f'\nBest val Dice : {best["val"]["dice"]:.4f} at epoch {best["epoch"]}')

if best_ckpt.exists():
    p = torch.load(best_ckpt, map_location='cpu')
    print(f'\nCheckpoint metadata:')
    print(f'  epoch      : {p.get("epoch")}')
    print(f'  best_dice  : {p.get("best_dice"):.4f}')
    print(f'  threshold  : {p.get("mask_threshold")}')
    print('\nUpdate configs/component4_lung.yaml → model.mask_threshold')
    print('with the value printed by the threshold sweep at the end of Cell 4.')

In [ ]:
# ── Cell 6: Per-dataset val Dice breakdown ────────────────────────────────────
import torch, sys, os
from pathlib import Path
from collections import defaultdict
from torch.utils.data import DataLoader

REPO_DIR   = Path('/kaggle/working/repo')
OUTPUT_DIR = Path('/kaggle/working/component4_out')
sys.path.insert(0, str(REPO_DIR))
os.chdir(str(REPO_DIR))

from src.components.component4_lung import Component4MedSAM
from src.data.component4_lung_dataset import (
    Component4LungDataset, collate_component4_batch, parse_manifest
)

device    = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
best_ckpt = OUTPUT_DIR / 'component4_mask_decoder.pt'
payload   = torch.load(best_ckpt, map_location=device)

model = Component4MedSAM(
    backend='auto',
    checkpoint_path=str(REPO_DIR / 'checkpoints/medsam/medsam_vit_b.pth'),
    model_type='vit_b',
    mask_threshold=float(payload.get('mask_threshold', 0.5)),
).to(device)
model.load_decoder_state_dict(payload['decoder_state_dict'])
model.eval()
print(f'Checkpoint: epoch={payload.get("epoch")}  best_dice={payload.get("best_dice"):.4f}')

val_records = parse_manifest(str(OUTPUT_DIR / 'component4_manifest.csv'),
                             repo_root=REPO_DIR, split='val')
val_loader  = DataLoader(
    Component4LungDataset(val_records, apply_clahe=None, augment=False),
    batch_size=2, num_workers=2, shuffle=False,
    collate_fn=collate_component4_batch,
)

threshold = float(payload.get('mask_threshold', 0.5))
smooth    = 1e-5
scores    = defaultdict(list)

with torch.no_grad():
    for batch in val_loader:
        x     = batch['x_3ch'].to(device)
        masks = batch['mask'].to(device)
        preds = (torch.sigmoid(model.forward_logits(x)) > threshold).float()
        for i, ds in enumerate(batch['dataset']):
            p, t = preds[i], masks[i]
            scores[ds].append(float((2*(p*t).sum()+smooth)/(p.sum()+t.sum()+smooth)))

print(f'\nPer-dataset val Dice (threshold={threshold}):')
print(f'{"Dataset":<15} {"N":>5} {"Mean":>10} {"Min":>8} {"Max":>8}')
print('─' * 50)
all_s = []
for ds, s in sorted(scores.items()):
    all_s.extend(s)
    print(f'{ds:<15} {len(s):>5} {sum(s)/len(s):>10.4f} {min(s):>8.4f} {max(s):>8.4f}')
print(f'{"Overall":<15} {len(all_s):>5} {sum(all_s)/len(all_s):>10.4f}')

## Next steps

1. **Download** `/kaggle/working/component4_out/component4_mask_decoder.pt`
2. **Upload** it to `mabdullahi454/tb-pipeline-checkpoints` (replace the old file)
3. **Update** `configs/component4_lung.yaml` → `model.mask_threshold` with the value from the threshold sweep at the end of Cell 4
4. **Re-run** `eval_baseline.ipynb` to confirm Lung Dice and Timika AUROC improvement

**If Shenzhen fails to load** (Cell 3 reports 0 images), inspect `shen_root contents` and verify the dataset slug is `iahmedhabib/shehzhenn`.